In [ ]:
from langchain_anthropic import ChatAnthropic
from dotenv import load_dotenv
import os

load_dotenv(override=True)

print(os.environ.get("ANTHROPIC_API_KEY"))  # Check if it's set

# Initialize the Claude model
llm = ChatAnthropic(
    model="claude-sonnet-4-6",             # Required: model ID to use
    temperature=1.0,                       # Randomness: 0.0 = deterministic, 1.0 = default (max varies by model)
    max_tokens=4096,                       # Max tokens in response (default from model profile, fallback 4096)
    top_k=None,                            # Top-K sampling: consider only top K tokens (None = disabled)
    top_p=None,                            # Top-P sampling: nucleus sampling threshold (None = disabled)
    timeout=None,                          # Request timeout in seconds (None = no timeout)
    max_retries=2,                         # Number of retries on failed requests
    stop=None,                             # Stop sequences e.g. ["\n", "END"] (None = disabled)
    base_url="https://api.anthropic.com",  # API base URL (can also set via ANTHROPIC_API_URL env var)
    streaming=False,                       # Stream tokens as generated (False = wait for full response)
    thinking=None,                         # Extended thinking e.g. {"type": "enabled", "budget_tokens": 5000}
)

# Send a simple message
response = llm.invoke("What is Agentic AI?")
print(response.content)

---
## `init_chat_model` — Provider-Agnostic Alternative

`init_chat_model` is a **factory function** from LangChain core that returns the right LLM class based on a string — so you are not locked to any one provider.

### ChatAnthropic vs init_chat_model

| | `ChatAnthropic` | `init_chat_model` |
|---|---|---|
| Import | `langchain_anthropic` | `langchain.chat_models` |
| Provider | Anthropic only | Any provider |
| Switch model | Rewrite import + class | Change a string |
| Type hints | Full Anthropic-specific | Generic |
| Best for | Single-provider projects | Multi-provider / configurable apps |

### How it works internally
```
init_chat_model('claude-sonnet-4-6', model_provider='anthropic')
         ↓
  returns ChatAnthropic(model='claude-sonnet-4-6')   ← same object!
```

In [ ]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv

load_dotenv(override=True)

# ── 1. Basic usage ──────────────────────────────────────────────────────────
llm = init_chat_model(
    model="claude-sonnet-4-6",
    model_provider="anthropic",   # tell LangChain which provider to use
    temperature=1.0,
    max_tokens=300,
)

response = llm.invoke("What is LangChain in one sentence?")
print("Claude:", response.content)
print()

# ── 2. Switch provider — only the string changes, rest of code is identical ─
# llm_openai = init_chat_model(
#     model="gpt-4o",
#     model_provider="openai",
#     temperature=1.0,
# )
# response = llm_openai.invoke("What is LangChain in one sentence?")
# print("GPT-4o:", response.content)

# ── 3. Configurable model — decided at runtime ───────────────────────────────
import os

# Imagine this comes from a config file or environment variable
MODEL_NAME     = os.getenv("LLM_MODEL",    "claude-sonnet-4-6")
MODEL_PROVIDER = os.getenv("LLM_PROVIDER", "anthropic")

dynamic_llm = init_chat_model(
    model=MODEL_NAME,
    model_provider=MODEL_PROVIDER,
    max_tokens=300,
)

r = dynamic_llm.invoke("Name one benefit of using LangChain.")
print(f"[{MODEL_PROVIDER}/{MODEL_NAME}]:", r.content)
print()

# ── 4. Confirm it returns the same underlying object as ChatAnthropic ────────
from langchain_anthropic import ChatAnthropic
print("Type via init_chat_model :", type(llm).__name__)           # ChatAnthropic
print("Type via ChatAnthropic   :", type(ChatAnthropic(model="claude-sonnet-4-6")).__name__)